In [ ]:
import os, sys, random, argparse
from texttable import Texttable
import pandas as pd
import torch
from torch_geometric.loader import DataLoader

sys.path.append(os.path.abspath('..'))

from Train import train
from AdaAqua.Data  import CustomDataset
from AdaAqua.Test  import test

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser() 

    parser.add_argument('--seed', type=int, default=10, help='Random seed of the experiment')
    parser.add_argument('--exp_name', type=str, default='Exp', help='Name of the experiment')

    parser.add_argument('--num_buildings_source', type=int, default=199, help='Number of buildings in the source domain')  
    parser.add_argument('--num_buildings_target', type=int, default=21, help='Number of buildings in the target domain')

    parser.add_argument('--num_epochs', type=int, default=20, help='Maximum number of epochs') 
    parser.add_argument('--sequence_len', type=int, default=96, help='Number of historical time steps') 
                        
    parser.add_argument('--input_dims', type=int, default=12, help='number of input dims') 
    parser.add_argument('--hidden_dims', type=int, default=24, help='number of hidden dims') 
    parser.add_argument('--output_dims', type=int, default=1, help='number of output dims')
    parser.add_argument('--layer_depth', type=int, default=5, help='number of Residual layers')

    parser.add_argument('--modes', type=int, default=16, help='Frequency modes of RFFT')
    parser.add_argument('--alpha', type=float, default=0.1, help='adversarial strength')

    parser.add_argument('--learning_rate', type=float, default=5e-6, help='Learning rate of Adam') 
    parser.add_argument('--gamma', type=float, default=0.95, help='decay parameter') 
    parser.add_argument('--weight_decay', type=float, default=1e-5, help='decay epoch') 
    parser.add_argument('--decay_epoch', type=float, default=1, help='decay epoch')
    parser.add_argument('--patience', type=int, default=10, help='patinece epoch') 
    
    # TODO For readers: adjust the dataset address to the location where you saved the Dataset.
    parser.add_argument('--source_data_path', type=str, default="demand_3d_ZZ.h5")
    parser.add_argument('--target_data_path', type=str, default="demand_3d_ZB.h5")
    parser.add_argument('--source_average_billing', type=str, default="ZZ_MEAN.csv")
    parser.add_argument('--target_average_billing', type=str, default="ZB_MEAN.csv")

    args = parser.parse_args(args=[])  

    args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    return args

In [ ]:
class IOStream():
    def __init__(self, path):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        self.file = open(path, 'a')

    def cprint(self, text):
        print(text)
        self.file.write(text + '\n')
        self.file.flush()

    def close(self):
        self.file.close()


def table_printer(args):
    args = vars(args) 
    keys = sorted(args.keys()) 
    table = Texttable()
    table.set_cols_dtype(['t', 't']) 
    rows = [["Parameter", "Value"]] 
    for k in keys:
        rows.append([k.replace("_", " ").capitalize(), str(args[k])]) 
    table.add_rows(rows)
    return table.draw()

In [ ]:
args = parse_args()

In [ ]:
if __name__ == '__main__':
    random.seed(args.seed)
    torch.manual_seed(args.seed)

    train_dataset = CustomDataset(args, dataset_type='train')
    val_dataset = CustomDataset(args, dataset_type='val')
    test_dataset = CustomDataset(args, dataset_type='test')
    train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=False, drop_last=False)
    val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False, drop_last=False)
    test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=False)

    # 3-fold cross validation
    for fold in range(3):
        
        args.fold = fold 

        output_dir_data = 'outputs_fold{}/DataManage'.format(fold+1) 
        output_dir_log_dir = 'outputs_fold{}/Log'.format(fold+1)        
        output_dir_log_file = '{}/log.txt'.format(output_dir_log_dir) 
        os.makedirs(output_dir_data, exist_ok=True) 
        os.makedirs(os.path.dirname(output_dir_log_dir), exist_ok=True)  

        IO = IOStream(output_dir_log_file)
        IO.cprint(str(table_printer(args)))  

        (train_loss1_list, train_loss2_list, val_loss_regress_list) = train(args, IO, train_dataloader, val_dataloader, output_dir_log_dir)
        (true_svae, pred_svae) = test(args, IO, test_dataloader, output_dir_log_dir) 

        true_svae  = pd.DataFrame(true_svae.T)
        pred_svae = pd.DataFrame(pred_svae.T)

        true_svae.to_csv(os.path.join(output_dir_data, 'true.csv'), index=False)
        pred_svae.to_csv(os.path.join(output_dir_data, 'pred.csv'), index=False)